# 01 · Keyword × colour overview

**What this notebook answers.** Which colours own which keyword mechanics, how strongly,
and whether that ownership has moved over the history of the game. Everything here is
built to support claims of the form *"60% of all Double Strike is red"* and *"the share of
red cards with Haste has been rising since 2015"* — and, crucially, to stop you from
confusing those two very different statements.

You need Magic knowledge to read this notebook. You do not need statistics knowledge —
every metric is explained where it is first used.

---

## Where the numbers come from

```
Scryfall bulk JSON          (data/raw/, downloaded once, cached 24h)
        │
        │  transform/   parse once: union multi-face cards, resolve colours,
        │               attach weights, order sets by release date
        ▼
5 parquet tables            (data/processed/)
   cards · card_colors · card_keywords · sets · color_totals
        │
        │  analysis/db.py   registers DuckDB views over the parquet
        ▼
3 views used by every chart below
   card_facts          one row per (card × colour)
   keyword_facts       one row per (card × colour × keyword)
   period_color_totals cards printed per (period × colour)  ← every denominator
        │
        ▼
   these charts
```

Nothing in this notebook re-reads the raw JSON. If a number looks wrong, the fix is
upstream in `transform/`, then `mtg-analysis build` again.

## Glossary

| Term | What it means here |
|---|---|
| `oracle_id` | One *gameplay* card. Lightning Bolt is a single `oracle_id` no matter how many times it has been printed — this is what stops reprint-heavy cards from dominating counts. |
| **period** | One set, running from its release until the next set's release. The x-axis of every time chart is set release dates, not calendar years. Sets can be grouped (see the last section). |
| `keyword_facts` | The main table for this notebook: one row per card × colour × keyword. A two-colour card with two keywords contributes four rows. |
| `period_color_totals` | How many cards each colour printed in each period. The denominator under every rate. |
| `paper_only` | The default filter. Excludes `set_type` values that are reprint or non-paper vehicles: `masters`, `memorabilia`, `funny`, `token`, `alchemy`, `minigame`, `promo`, `treasure_chest`. Pass `set_type_filter="unfiltered"` to any function to include them. |
| **weighting** | How a gold (multicolour) card is attributed to its colours. See below — this is the single most important idea in the project. |

## The two weightings

A mono-red card is easy: it is one red card. A Boros card is the problem — is it a red
card, a white card, or half of each? Both answers are useful, so both are stored and every
function takes a `weighting` argument. For a card with $n$ colours, each of its colours $c$
gets:

$$w_{\text{frac}}(c) = \frac{1}{n} \qquad\qquad w_{\text{incl}}(c) = 1$$

**Worked example.** Two cards with Haste: a mono-red one, and a red-white one.

| Weighting | Red | White | Total | Reads as |
|---|---|---|---|---|
| `fractional` | $1 + 0.5 = 1.5$ | $0.5$ | $2.0$ | one unit per card — shares add up to 100% |
| `inclusive` | $1 + 1 = 2$ | $1$ | $3.0$ | "cards that touch this colour" — the gold card counted twice |

Use **fractional** for "what share of this keyword is red" (the pie must add to 100%).
Use **inclusive** for "how many red cards have this" (a Boros card really is a red card
you can cast). Colourless cards get a single `C` row with both weights at 1.0.

Nothing in this project picks one silently — output always states which was used.

---

Run `mtg-analysis fetch` and `mtg-analysis build` before this notebook; it reads only the
built parquet tables.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from mtg_analysis.analysis.db import get_connection
from mtg_analysis.analysis.design_volume import design_volume, plot_design_volume
from mtg_analysis.analysis.heatmap import (
    keyword_color_matrix,
    pivot_matrix,
    plot_keyword_color_heatmap,
)
from mtg_analysis.analysis.timeseries import keyword_timeseries, plot_keyword_timeseries
from mtg_analysis.config import load_config
from mtg_analysis.metrics.core import color_share, penetration_rate, trend_report

config = load_config("../config/config.yaml")
con = get_connection("../" / config.paths.processed_dir, config.periods)
con.execute("SELECT COUNT(*) AS cards FROM cards").pl()

## 1 · Keyword × colour heatmap

**What it measures.** The colour pie for every keyword at once — the single "which colours
own what" picture, across all of Magic's history.

**Where the data comes from.** `keyword_facts`, filtered to `paper_only`, grouped by
`(keyword, colour)`. Two numbers come back per cell: `raw_count` (how many cards) and
`weight` (the same cards, weighted). `top_n=20` keeps the twenty most-used keywords so the
grid stays readable.

**The calculation.** For keyword $K$ and colour $C$, the share is that colour's weight over
the keyword's total weight across all colours:

$$\text{share}(K, C) = \frac{\sum w(K, C)}{\sum_{c} w(K, c)}$$

In words: *of all the Haste in Magic, how much of it is red?*

**Worked example**, continuing the two-card toy from the primer:

- fractional → red share $= 1.5 / 2.0 = \mathbf{75\%}$
- inclusive → red share $= 2 / 3 = \mathbf{67\%}$

Same two cards, two defensible answers. The heatmap defaults to fractional, so each row
adds to 100%.

**How to read it.** Rows are keywords, columns are colours, and *each row sums to 100%*.
So the heatmap compares colours **within** a keyword — a dark cell means "this colour owns
this mechanic", not "this keyword is common". Deathtouch being 97% black tells you nothing
about how many Deathtouch cards exist; that is what the `raw_count` table underneath is
for.

**One caveat about the table below the chart.** The `keyword_count` column sums per-colour
card counts, so a Boros card is counted once for red and once for white. It is the ranking
key behind `top_n` — do not read it as "number of cards with this keyword".

In [ ]:
matrix = keyword_color_matrix(con, top_n=20)
plot_keyword_color_heatmap(matrix, value="share")
plt.show()
pivot_matrix(matrix, value="raw_count")

## 2 · The two headline numbers for one keyword

Before plotting anything over time, get the two summary numbers straight, because they
answer different questions and are the easiest thing in this project to mix up.

**Colour share** — the slice of the keyword's pie held by this colour:

$$\text{share}(K, C) = \frac{\sum w(K, C)}{\sum_{c} w(K, c)}$$

> *Of all the Double Strike in Magic, how much of it is red?*

**Penetration rate** — how saturated that colour is with the keyword, using
`period_color_totals` as the denominator:

$$\text{penetration}(K, C) = \frac{\sum w(K, C)}{\text{colorTotal}(C)}$$

> *Of all the red cards ever printed, how many have Double Strike?*

**Worked example.** Take the two-card toy again (red weight for Haste $= 1.5$) and suppose
red printed 10 fractional-weighted cards in total that period. Share was 75%; penetration
is $1.5 / 10 = \mathbf{15\%}$. A colour can hold most of a keyword (high share) while
almost none of its cards actually have it (low penetration) — that is normal for niche
mechanics.

**Reading the output.** The cell prints both metrics under both weightings, labelled. A
`NaN` means the denominator was zero — no data, which is *not* the same as 0%.

In [ ]:
keyword, color = "Double strike", "R"
for weighting in ("fractional", "inclusive"):
    share = color_share(con, keyword, color, weighting=weighting)
    rate = penetration_rate(con, keyword, color, weighting=weighting)
    print(f"{weighting:>10}: {color} holds {share:.1%} of {keyword}; "
          f"{rate:.2%} of {color} cards carry it")

## 3 · Trend over sets

**What it measures.** How a keyword's relationship with a colour has moved across Magic's
history, set by set.

**Where the data comes from.** `trend_report` joins two things per period: the keyword
weights from `keyword_facts`, and the colour's total from `period_color_totals`. The join
runs *from the totals side* (a `LEFT JOIN`), so a set where the keyword never appeared
still shows up as a row with `raw_count = 0` rather than silently vanishing from the
series. One row = one set.

**The columns.**

| Column | Meaning |
|---|---|
| `period` | The set code (periods are sets, in release order) |
| `raw_count` | Unweighted headcount of cards of that colour with the keyword |
| `color_share` | This colour's slice of the keyword, in that set |
| `penetration_rate` | Share of that colour's cards in that set carrying the keyword |
| `color_total` | The denominator behind `penetration_rate` — always shown, never implied |
| `weighting`, `set_type_filter` | Which scheme and filter produced the row |

All of `raw_count`, `color_share` and `penetration_rate` come back in the same frame
deliberately: a raw count without its denominator is not a finding.

In [ ]:
trend_report(con, "Haste", "R").tail(10)

### Why both panels are on screen

The top panel is **penetration** (is this colour actually printing the mechanic more?) and
the bottom is **share** (is this colour's slice of the mechanic growing?). They can move in
*opposite directions*, and reading only one is the most likely way to publish something
false from this dataset.

**Worked example.**

| | Red cards printed | Red with the keyword | White with the keyword | Red share | Red penetration |
|---|---|---|---|---|---|
| Era A | 10 | 2 | 6 | $2/8 = 25\%$ | $2/10 = 20\%$ |
| Era B | 10 | 1 | 0 | $1/1 = 100\%$ | $1/10 = 10\%$ |

Red's **share went 25% → 100%** while its **penetration halved**. Nothing about red became
more hasty; every other colour simply stopped printing the mechanic. "Red now owns Double
Strike" would be true-but-misleading, and "red is getting more Double Strike" would be flat
wrong.

**How to read the charts.** X-axis is set release date, one marker per set. Each colour has
its own dash pattern and marker as well as its hue, so the series stay separable in
greyscale and for colourblind readers. Jaggedness is expected — a 30-card supplemental set
and a 300-card expansion are each a single point. The last section of this notebook
smooths that.

In [ ]:
series = keyword_timeseries(con, "Haste", colors=["W", "U", "B", "R", "G"])
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
plot_keyword_timeseries(series, metric="penetration_rate", ax=axes[0])
plot_keyword_timeseries(series, metric="color_share", ax=axes[1])
fig.tight_layout()
plt.show()

## 4 · Design volume — the denominator itself

**What it measures.** How many cards each colour printed in each set. This is not a
side-note chart: it is the denominator underneath every rate above, plotted directly.

**Where the data comes from.** `period_color_totals`, which is `color_totals` rolled up to
the current period grouping. Under fractional weighting the colours in one set sum to the
number of cards in that set (each card contributes exactly 1.0 spread across its colours);
under inclusive weighting they sum higher, because gold cards are counted in each of their
colours.

**Why you need it.** A raw-count chart that rises in 2019 may only be telling you 2019 was
a big year for Magic. Print volume has changed enormously over the game's life, so any
claim of the form "there are more X now" has to be checked against this chart before it
means anything.

**How to read it.** Flat-ish lines with occasional spikes (large sets) and dips (small
supplemental ones). Colourless sits well below the five colours — expected, since only
artifacts, lands and eldrazi land there.

In [ ]:
volume = design_volume(con)
plot_design_volume(volume)
plt.show()
volume.tail()

## 5 · Smoothing noisy sets

Per-set periods are the finest grain available, and at real data density (~900 sets) they
are very noisy: one small set with three relevant cards produces a wild swing in a rate.

`PeriodGroupConfig(mode="rolling_sets", group_size=5)` groups five consecutive sets — by
`set_order`, which is release date with set code as tiebreak — into one period labelled
`firstset..lastset`. Each grouped period's release date is its earliest set's.

**The important part:** grouping is applied **at query time**. `color_totals` stays
materialized at per-set grain, so switching grouping costs one function call and never
requires re-running `mtg-analysis build`. Try `group_size=10` for a decade-ish view.

In [ ]:
from mtg_analysis.analysis.db import set_period_config
from mtg_analysis.config import PeriodGroupConfig

set_period_config(con, PeriodGroupConfig(mode="rolling_sets", group_size=5))
plot_keyword_timeseries(keyword_timeseries(con, "Haste", colors=["R", "W"]))
plt.show()
set_period_config(con, config.periods)

---

# Write-up · what I found

Fill this in as you explore. The prompts exist to keep a claim honest.

### Claim

> *e.g. "Red's share of Double Strike has grown since 2015."*

### Evidence

| | Value | Weighting | Filter |
|---|---|---|---|
| Colour share, start of window | | | |
| Colour share, end of window | | | |
| Penetration, start of window | | | |
| Penetration, end of window | | | |
| Colour total (denominator) trend | | | |

### Checks before believing it

- [ ] Share **and** penetration reported together — do they agree, or diverge?
- [ ] Denominator checked: is the colour's print volume itself moving?
- [ ] Claim re-run under the other weighting — does it survive `inclusive`?
- [ ] Set filter stated (`paper_only` by default); does `unfiltered` change it?
- [ ] Enough cards per period to be real, or is this small-sample noise?

### Conclusion